# ChessHack — all-in-one (48 cores + 80GB GPU)

Two paths on one machine: **(A)** distill from Stockfish (Phase 1: label on CPU → distill-train on GPU) then self-play, or **(B) pure self-play from scratch** (skip Phase 1, jump to Phase 2). Self-play = CPU workers + GPU inference server. Checkpoints/shards/Elo persist to Google Drive so a session restart loses nothing.

**Before running:** set `REPO_URL` (cell 4) to your GitHub repo. Runtime → Change runtime type → **GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
import os; print('CPU cores:', os.cpu_count())

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE='/content/drive/MyDrive/chesshack'
for d in ('data','data/distill','data/nets','bench'): os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
print('persisting to', DRIVE)

In [ ]:
import os
REPO_URL='https://github.com/anilmercanoglu-debug/chesshack.git'   # <-- set this
if not os.path.isdir('/content/chesshack/.git'):
    !git clone $REPO_URL /content/chesshack
else:
    !cd /content/chesshack && git pull
%cd /content/chesshack
!git log --oneline -1

In [ ]:
# deps: torch ships with Colab (CUDA). add python-chess + zstandard (stream .pgn.zst); fetch Stockfish into tools/
!pip -q install python-chess zstandard
import os, urllib.request, tarfile, shutil, glob
os.makedirs('tools', exist_ok=True)
if not os.path.exists('tools/stockfish'):
    url='https://github.com/official-stockfish/Stockfish/releases/latest/download/stockfish-ubuntu-x86-64-avx2.tar'
    urllib.request.urlretrieve(url,'/tmp/sf.tar'); tarfile.open('/tmp/sf.tar').extractall('/tmp')
    b=[f for f in glob.glob('/tmp/stockfish/*') if os.path.isfile(f) and os.access(f,os.X_OK)][0]
    shutil.copy(b,'tools/stockfish'); os.chmod('tools/stockfish',0o755)
import torch; print('torch',torch.__version__,'cuda',torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!echo 'sanity:'; echo -e 'uci\nquit' | ./tools/stockfish | grep '^id name'

In [ ]:
# persist data/ + bench/ to Drive (survive session restarts)
import os, shutil
for d in ('data','bench'):
    local=f'/content/chesshack/{d}'; target=f'{DRIVE}/{d}'
    if os.path.islink(local): continue
    if os.path.isdir(local): shutil.copytree(local,target,dirs_exist_ok=True); shutil.rmtree(local,ignore_errors=True)
    os.makedirs(target,exist_ok=True); os.symlink(target,local)
!ls -la data/distill | head

## Phase 1a — label REAL-GAME positions with Stockfish (uses ALL cores)
Sources positions from the **Lichess Elite DB** (both players 2400+) so the net learns the positions strong play actually reaches, then relabels each with Stockfish (policy = MultiPV softmax, value = eval). ~200+ pos/s on 48 cores → ~800k positions in ~1h. Writes to a **new** dir (`data/distill_real`), leaving any old synthetic shards + nets untouched.

In [ ]:
import os, urllib.request, zipfile, glob
# Lichess Elite DB: monthly .zip of 2400+ vs 2400+ games -> realistic strong-play positions.
# (distill relabels with Stockfish, so the game RESULT is unused -- we only want the positions.)
PGN_MONTH='2023-12'   # pick any available month from database.nikonoel.fr
if not glob.glob('/content/elite/*.pgn'):
    zp=f'/content/elite_{PGN_MONTH}.zip'
    urllib.request.urlretrieve(f'https://database.nikonoel.fr/lichess_elite_{PGN_MONTH}.zip', zp)
    zipfile.ZipFile(zp).extractall('/content/elite')
PGN=glob.glob('/content/elite/*.pgn')[0]
print('PGN:', PGN, os.path.getsize(PGN)//1_000_000,'MB')
# 85% of positions from real games (rest synthetic for off-book/sparse-endgame coverage) -> NEW dir
!python -m trainer.label --n 800000 --pgn "{PGN}" --out data/distill_real --seed 1 --workers {os.cpu_count()}

## Phase 1b — distillation training (GPU, production net C256/B20)
Trains **from scratch** on the real-game shards → `distilled_real.pt` (does NOT touch `champion.pt`/`distilled.pt`). Watch `top1` (how often the net's argmax matches Stockfish's best move) climb — that is the prior getting sharp. Re-run to continue.

In [ ]:
!python -m trainer.train --mode distill --data data/distill_real --net prod --steps 40000 --batch 1024 --lr 2e-3 --workers 16 --out data/nets/distilled_real.pt

In [ ]:
# raw policy first -- the prior self-play must raise (was ~997 on the synthetic-distilled net; want >=1400)
!python -m bench.policy_elo --ckpt data/nets/distilled_real.pt --games 40
# then MCTS Elo vs the SF ladder + anti-stall sanity (search_gain)
!python -m bench.elo --ckpt data/nets/distilled_real.pt --games 40 --sims 800
!python -m bench.search_gain --ckpt data/nets/distilled_real.pt --sims 600 --games 40

## Phase 1c — capacity test (SCALE net, ~76M)
Distill the larger SCALE net (C384/B28, ~76M) on the **same** `distill_real` data, then compare to PROD (`distilled_real.pt`). If SCALE clearly beats PROD (higher `top1` / raw-Elo / MCTS-Elo) → the 24.5M net was **capacity-bound**, growing helps. If SCALE ≈ PROD → the ceiling is the **labels** (SF@100k nodes), not size → re-label stronger instead. ~2h (3× PROD); the bench uses `data/openings.txt` if present.

PROD reference: raw policy **1126** · MCTS@openings **0.925 / 0.688 / 0.575 / 0.550** (SF 1320/1500/1700/1900).

In [ ]:
import os
OP = '--openings data/openings.txt' if os.path.exists('data/openings.txt') else ''
# distill the 76M net from scratch on the same real-game shards
!python -m trainer.train --mode distill --data data/distill_real --net scale \
  --steps 40000 --batch 1024 --lr 2e-3 --workers 16 --out data/nets/distilled_scale.pt
# head-to-head bench vs PROD references
!python -m bench.policy_elo --ckpt data/nets/distilled_scale.pt --games 40
!python -m bench.elo --ckpt data/nets/distilled_scale.pt --sims 1600 --games 40 {OP}

## Phase 2 — pure self-play from scratch (AlphaZero-style)
Start from a **random 76M net** (no distillation) and let self-play + MCTS bootstrap it. Sims **ramp from 50** (auto-bumped as the net catches its own search), up to 1600; **gate every 1000 games** (50-game match); real SF-anchored **Elo checkup every 100k games** (→ `bench/elo_history.json`). **You can skip Phase 1 entirely** (labeling/distill) — run cells 1–5, then this.

Honest note: from scratch on one GPU is the slow road — early checkups show **~0 Elo** (random net still learning); watch the **trend**, not single numbers. Expect days–weeks to strong play, and a 76M net bootstraps slower than a small one.

In [ ]:
# opening book: real opening positions from the Elite PGN to SEED self-play from (anti echo-chamber).
# Persists to Drive via data/; only needs to run once. Re-downloads the PGN if /content was wiped.
import os, glob, random, urllib.request, zipfile
from trainer.positions import opening_fens
if not os.path.exists('data/openings.txt'):
    pgns = glob.glob('/content/elite/**/*.pgn', recursive=True)
    if not pgns:                                   # fresh session -> /content wiped, re-download
        PGN_MONTH='2023-12'
        zp=f'/content/elite_{PGN_MONTH}.zip'
        urllib.request.urlretrieve(f'https://database.nikonoel.fr/lichess_elite_{PGN_MONTH}.zip', zp)
        zipfile.ZipFile(zp).extractall('/content/elite')
        pgns = glob.glob('/content/elite/**/*.pgn', recursive=True)
    ops = opening_fens(pgns[0], 50000, random.Random(0))   # ~position at ply 8-16 of each game
    open('data/openings.txt','w').write('\n'.join(ops)+'\n')
    print('wrote', len(ops), 'opening FENs to data/openings.txt')
else:
    print('data/openings.txt exists:', sum(1 for _ in open('data/openings.txt')), 'openings')

In [ ]:
import os
W=max(2, os.cpu_count()-4)
# PURE self-play from a RANDOM 76M net (no distillation). sims auto-ramp 50->1600.
# First run: NO --resume. After it has checkpointed once, re-run WITH --resume to survive disconnects.
!python -m trainer.selfplay --init-from scratch --net scale --workers {W} \
  --steps 100000000 --batch 1024 --lr 2e-4 --sims 50 \
  --base-elo 0 --bench-every-games 100000 \
  --gate-every-games 1000 --gate-games 50
# Distilled-base alternative (if you run Phase 1 instead):
#   --init-from data/nets/distilled_real.pt --net prod --sims 600 --base-elo 1800 \
#   --bench-every-promos 3 --openings data/openings.txt --random-open-frac 0.5

## Notes
- **Resume:** re-run cells 1–5 in a new session; Drive has the checkpoints. Re-run Phase 2 **with `--resume`** to continue.
- **Monitor Elo:** `import json; print(json.load(open('bench/elo_history.json'))[-1])`.
- **Net sizes:** `--net dev` (10.7M) / `prod` (24.5M) / `scale` (76M). `--init-from scratch` builds the chosen size with **random** weights.
- Free Colab disconnects ~12h; Drive persistence means no progress lost.
- 80GB lets you raise `--batch` and self-play `--workers`; watch GPU util + `avg_batch` in the logs.

## Play against the bot (browser UI)
Launches the web board and prints a clickable URL (Colab proxy). Use `champion.pt` once self-play has run, else `distilled.pt`. Drag pieces to move.

In [ ]:
import subprocess, time, os
ckpt=next(p for p in ('data/nets/champion.pt','data/nets/distilled_real.pt','data/nets/distilled.pt') if os.path.exists(p))
proc=subprocess.Popen(['python','serve.py','--ckpt',ckpt,'--sims','800','--port','8000'])
time.sleep(8)
from google.colab.output import eval_js
print('bot:',ckpt,'\nOpen and play:', eval_js('google.colab.kernel.proxyPort(8000)'))